# Inspecting value pipelines in an interactive simulation

Most of what you see about a simulant in a Vivarium simulation is produced by a
**value pipeline**: a component provides a *source* for the value, other components
attach *modifiers* to it, a *combiner* says how the modifiers are applied, and optional
*post-processors* finish the job. The child wasting exposure in the
[tutorial](../../../onboarding_resources/tutorial/index.ipynb) is a good example:
the `Risk` component produces it, and the tutorial's `SQLNS` intervention modifies it.

Starting with `vivarium-engine` 5.10, the interactive simulation lets you look at
these pipelines directly:

* `sim.get_attribute(name)` returns the pipeline object. It is still recommended to get
  attribute *values* with `sim.get_population()`, but the pipeline itself is useful for
  debugging.
* Printing a pipeline shows its source, combiner, and all of its modifiers
  (and post-processors).
* Sources, combiners, and modifiers print descriptively on their own, too.
* When a component registers a pipeline or a modifier, it can pass an optional
  `description`, which also shows up in the prints.

This page demonstrates each of these on the SQ-LNS model from the
[tutorial](../../../onboarding_resources/tutorial/index.ipynb). It is a companion to
the [interactive simulation page](index.rst), which covers setting up an interactive
simulation in general.

## Setting up the simulation

We start the same way as the tutorial: quiet down the debugging output, then build the
model out of `vivarium_public_health` components plus a custom intervention component.

In [1]:
import sys
import warnings

import pandas as pd

warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

from loguru import logger as log

log.remove()
log.add(sys.stderr, level="ERROR")

pd.options.display.max_rows = 12

In [2]:
import gbd_mapping
import vivarium.public_health
from vivarium.engine import Component, InteractiveContext
from vivarium.engine.framework.engine import Builder
from vivarium.engine.framework.population import SimulantData

# The tutorial's data artifact. Edit this path if you downloaded the files somewhere else.
ARTIFACT_PATH = "../../../onboarding_resources/tutorial/tutorial.hdf"

### Components

These are the components from the tutorial, with two changes needed for current
versions of `vivarium_public_health`:

* `Mortality` no longer needs to be listed separately, because `BasePopulation` now
  brings it along.
* Components no longer declare the columns they create with a `columns_created`
  attribute. Instead they register an *initializer* for those columns, and they modify
  attribute pipelines with `register_attribute_modifier`.

The other change is the one this page is about: when we register our modifier, we pass a
`description` of what it does.

First, a reminder of what the child wasting categories mean, using `gbd_mapping` as in
the tutorial. The intervention moves children in `cat1` and `cat2` to `cat3`.

In [3]:
categories = gbd_mapping.risk_factors.child_wasting.categories.to_dict()
categories

{'cat1': 'Severe Wasting, < -3 SD (post-ensemble)',
 'cat2': 'Wasting Between -3 SD and -2 SD (post-ensemble)',
 'cat3': 'Wasting Between -2 SD and -1 SD (post-ensemble)',
 'cat4': 'Unexposed'}

In [4]:
class SQLNS(Component):
    """Delivers SQ-LNS to a configurable share of simulants, moving some of the
    moderately and severely wasted children who receive it into the mild category."""

    CONFIGURATION_DEFAULTS = {"sqlns": {"coverage": 0.0}}

    def setup(self, builder: Builder) -> None:
        self.coverage = builder.configuration["sqlns"]["coverage"]
        self.randomness = builder.randomness.get_stream("sqlns")
        # This declares the two columns this component creates and how to fill them
        # when simulants are initialized.
        builder.population.register_initializer(
            initializer=self.on_initialize_simulants,
            columns=["covered_by_sqlns", "would_benefit_from_sqlns"],
            required_resources=[self.randomness],
        )
        # child_wasting.exposure is an attribute pipeline created by the Risk component.
        # We attach a modifier to it, and describe what the modifier does.
        builder.value.register_attribute_modifier(
            "child_wasting.exposure",
            modifier=self.intervention_effect,
            required_resources=["covered_by_sqlns", "would_benefit_from_sqlns"],
            description=(
                "Move covered simulants who benefit from SQ-LNS out of moderate or "
                "severe wasting (cat1, cat2) into mild wasting (cat3)"
            ),
        )

    def on_initialize_simulants(self, pop_data: SimulantData) -> None:
        covered_by_sqlns = (
            self.randomness.get_draw(pop_data.index, additional_key="covered_by_sqlns")
            <= self.coverage
        )
        # 13% of those who receive SQ-LNS and are wasted benefit from it, in that they
        # are no longer wasted (see the tutorial for where this number comes from).
        would_benefit_from_sqlns = (
            self.randomness.get_draw(pop_data.index, additional_key="would_benefit_from_sqlns")
            <= 0.13
        )
        self.population_view.initialize(
            pd.DataFrame(
                {
                    "covered_by_sqlns": covered_by_sqlns,
                    "would_benefit_from_sqlns": would_benefit_from_sqlns,
                },
                index=pop_data.index,
            )
        )

    def intervention_effect(self, index: pd.Index, child_wasting: pd.Series) -> pd.Series:
        # 'index' is the simulants being asked about, and 'child_wasting' holds their
        # exposure categories as produced by the earlier stages of the pipeline.
        pop = self.population_view.get(index, ["covered_by_sqlns", "would_benefit_from_sqlns"])
        moderate_or_severe = child_wasting.isin(["cat1", "cat2"])
        benefits = moderate_or_severe & pop["covered_by_sqlns"] & pop["would_benefit_from_sqlns"]
        child_wasting = child_wasting.copy()
        child_wasting[benefits] = "cat3"
        return child_wasting

We also add one small component that is *not* in the tutorial. It registers a new
attribute pipeline, `child_wasting.is_wasted`, which is `True` for simulants who are
moderately or severely wasted. We will use it to summarize wasting in each scenario, and
it lets us show what a `description` looks like on a pipeline (rather than on a modifier).

In [5]:
class WastingStatus(Component):
    """Adds a True/False attribute for whether a simulant is wasted (WHZ below -2)."""

    def setup(self, builder: Builder) -> None:
        builder.value.register_attribute_producer(
            "child_wasting.is_wasted",
            source=self.is_wasted,
            required_resources=["child_wasting.exposure"],
            description="True for simulants in the moderate or severe wasting categories (WHZ < -2)",
        )

    def is_wasted(self, index: pd.Index) -> pd.Series:
        exposure = self.population_view.get(index, "child_wasting.exposure")
        return exposure.isin(["cat1", "cat2"])

In [6]:
def make_components():
    return [
        # Demographics (Mortality is included in BasePopulation)
        vivarium.public_health.BasePopulation(),
        vivarium.public_health.FertilityCrudeBirthRate(),
        # Cause
        vivarium.public_health.SIS("diarrheal_diseases"),
        # Risk, and its effect on the cause
        vivarium.public_health.Risk("risk_factor.child_wasting"),
        vivarium.public_health.RiskEffect(
            "risk_factor.child_wasting",
            "cause.diarrheal_diseases.incidence_rate",
        ),
        # Observers
        vivarium.public_health.DisabilityObserver(),
        vivarium.public_health.MortalityObserver(),
        # Our own components
        SQLNS(),
        WastingStatus(),
    ]

### Configuration and scenarios

The configuration is the tutorial's, except that we use a much smaller population so
that this page runs quickly. As in the tutorial, we set up a baseline scenario with no
SQ-LNS coverage and an intervention scenario with full coverage.

In [7]:
configuration = {
    "input_data": {
        "artifact_path": ARTIFACT_PATH,
        "input_draw_number": 0,
    },
    "time": {
        "step_size": 7,
        "start": {"year": 2025, "month": 1, "day": 1},
        "end": {"year": 2030, "month": 12, "day": 31},
    },
    "population": {
        "initialization_age_min": 0,
        "initialization_age_max": 2,
        "population_size": 10_000,
        "untracking_age": 2,
    },
}

sim_baseline = InteractiveContext(
    components=make_components(),
    configuration={"sqlns": {"coverage": 0}, **configuration},
)
sim_intervention = InteractiveContext(
    components=make_components(),
    configuration={"sqlns": {"coverage": 1}, **configuration},
)

## Which attributes are there?

In the tutorial, we looked at `sim.get_population()` to see the columns of the
population table and at `sim.list_values()` to see the value pipelines. In the current
`vivarium-engine`, columns and pipelines are unified as **attributes**, and
`get_attribute_names()` lists all of them.

In [8]:
sim_intervention.get_attribute_names()

['age',
 'sex',
 'location',
 'entrance_time',
 'exit_time',
 'is_aged_out',
 'cause_specific_mortality_rate',
 'affected_unmodeled.cause_specific_mortality_rate',
 'mortality_rate',
 'is_alive',
 'cause_of_death',
 'years_of_life_lost',
 'all_causes.disability_weight',
 'diarrheal_diseases',
 'susceptible_to_diarrheal_diseases.prevalence',
 'susceptible_to_diarrheal_diseases.birth_prevalence',
 'susceptible_to_diarrheal_diseases.dwell_time',
 'susceptible_to_diarrheal_diseases_event_time',
 'susceptible_to_diarrheal_diseases_event_count',
 'diarrheal_diseases.incidence_rate',
 'diarrheal_diseases.prevalence',
 'diarrheal_diseases.birth_prevalence',
 'diarrheal_diseases.dwell_time',
 'diarrheal_diseases_event_time',
 'diarrheal_diseases_event_count',
 'diarrheal_diseases.disability_weight',
 'diarrheal_diseases.excess_mortality_rate',
 'diarrheal_diseases.remission_rate',
 'child_wasting.exposure_distribution.ppf',
 'risk_factor.child_wasting.exposure_parameters',
 'child_wasting.expos

`list_values()` still exists, but it now only lists the few value pipelines that are not
simulant attributes (for example, calibration constants).

In [9]:
sim_intervention.list_values()

['simulant_step_size',
 'affected_unmodeled.cause_specific_mortality_rate.calibration_constant',
 'diarrheal_diseases.incidence_rate.calibration_constant',
 'diarrheal_diseases.excess_mortality_rate.calibration_constant',
 'diarrheal_diseases.remission_rate.calibration_constant']

## Getting a pipeline object

`get_attribute()` returns the pipeline behind an attribute. When you echo it in a
notebook cell, you get a full description of how the value is put together.

In [10]:
exposure = sim_intervention.get_attribute("child_wasting.exposure")
exposure

child_wasting.exposure  [attribute pipeline]
registered by   risk_factor.child_wasting

source          child_wasting.exposure_distribution.ppf (attributes)
combiner        replace_combiner
modifiers       1 (order not guaranteed)
                  - SQLNS.intervention_effect
                     from sqlns
                     Move covered simulants who benefit from SQ-LNS out of moderate or severe wasting (cat1, cat2) into mild wasting (cat3)
post-processors none

Reading this from the top:

* **registered by**: the component that created the pipeline. Here that is the `Risk`
  component, whose name is `risk_factor.child_wasting`.
* **source**: where the starting value comes from. This pipeline's source is another
  attribute, `child_wasting.exposure_distribution.ppf`, which the risk's exposure
  distribution produces from each simulant's propensity.
* **combiner**: how each modifier's output is combined with the value so far.
  `replace_combiner` means each modifier receives the current value and returns a
  replacement for it.
* **modifiers**: every modifier attached to the pipeline, with the component it came
  from and, when one was given, its description. Our `SQLNS.intervention_effect` is
  listed here, along with the description we wrote when we registered it.
* **post-processors**: transformations applied after all modifiers have run. There are
  none here.

Because this is the intervention scenario, we know our modifier is doing something. But
the print shows *structure*, not configuration: the baseline scenario prints exactly the
same thing, since the `SQLNS` component is present there too, just with zero coverage.

## `repr()`, echoing, and `print()`

Echoing a pipeline in a notebook shows this full description rather than the usual
one-line `repr`. If you want the short version, call `repr()` explicitly:

In [11]:
repr(exposure)

"AttributePipeline('child_wasting.exposure')"

`print()` gives the full description, in a notebook or anywhere else. The difference
only matters in the plain Python REPL, where echoing shows the `repr` and you need
`print()` to see the description.

In [12]:
print(exposure)

child_wasting.exposure  [attribute pipeline]
registered by   risk_factor.child_wasting

source          child_wasting.exposure_distribution.ppf (attributes)
combiner        replace_combiner
modifiers       1 (order not guaranteed)
                  - SQLNS.intervention_effect
                     from sqlns
                     Move covered simulants who benefit from SQ-LNS out of moderate or severe wasting (cat1, cat2) into mild wasting (cat3)
post-processors none


## The pieces of a pipeline print descriptively too

The pipeline's source, combiner, modifiers, and post-processors are all available as
attributes of the pipeline object, and each of them prints a readable description.

In [13]:
exposure.source

child_wasting.exposure_distribution.ppf (attributes)

In [14]:
exposure.combiner

replace_combiner

Modifiers are stored in a list called `mutators`. They are listed in the order they are
currently applied, but as the print says, that order is not guaranteed: it depends on the
order in which components were set up, so don't write code that relies on it.

In [15]:
exposure.mutators

[SQLNS.intervention_effect from sqlns
Move covered simulants who benefit from SQ-LNS out of moderate or severe wasting (cat1, cat2) into mild wasting (cat3)]

In [16]:
exposure.mutators[0]

SQLNS.intervention_effect from sqlns
Move covered simulants who benefit from SQ-LNS out of moderate or severe wasting (cat1, cat2) into mild wasting (cat3)

In [17]:
exposure.post_processor

[]

## Using the pipeline object for debugging

For the *values* of an attribute, keep using `get_population()`. It now takes the names
of the attributes you want, and pipelines and columns can be mixed freely:

In [18]:
sim_intervention.get_population(
    ["age", "sex", "child_wasting.exposure", "covered_by_sqlns", "would_benefit_from_sqlns"]
)

,age,sex,child_wasting.exposure,covered_by_sqlns,would_benefit_from_sqlns
0,0.449100,Female,cat4,True,False
1,0.298315,Male,cat1,True,False
2,0.041546,Female,cat4,True,True
3,0.655818,Male,cat4,True,False
4,0.563762,Female,cat3,True,False
...,...,...,...,...,...
9995,1.536697,Female,cat4,True,False
9996,0.620302,Female,cat4,True,True
9997,0.702214,Female,cat3,True,False
9998,0.357439,Female,cat4,True,True


The pipeline object can also be called directly with an index of simulants, like in the
tutorial. What makes it useful for debugging is the `mode` argument: `mode="source"`
evaluates only the source, skipping all modifiers, so you can see exactly what the
modifiers changed. Here, our intervention moves some children from the severe and
moderate categories into the mild one, and leaves the unexposed alone, which is exactly
what it is supposed to do.

In [19]:
index = sim_intervention.get_population_index()
pd.DataFrame(
    {
        "before modifiers (mode='source')": exposure(index, mode="source").map(categories).value_counts(),
        "after modifiers": exposure(index).map(categories).value_counts(),
    }
)

,before modifiers (mode='source'),after modifiers
child_wasting.exposure,,
Unexposed,6860,6860
Wasting Between -2 SD and -1 SD (post-ensemble),2002,2140
Wasting Between -3 SD and -2 SD (post-ensemble),825,724
"Severe Wasting, < -3 SD (post-ensemble)",313,276


## A pipeline with several modifiers

The value that our intervention ultimately aims to change is the incidence rate of
diarrheal diseases. Its pipeline is more involved: it uses a `multiplication_combiner`,
it has two modifiers, and it has a post-processor that rescales the yearly rate to the
size of the time step.

In [20]:
sim_intervention.get_attribute("diarrheal_diseases.incidence_rate")

diarrheal_diseases.incidence_rate  [attribute pipeline]
registered by   _risk_affected_pipeline.diarrheal_diseases.incidence_rate

source          RateTransition.compute_transition_rate (callable)
combiner        multiplication_combiner
modifiers       2 (order not guaranteed)
                  - _RiskAffectedPipeline._apply_calibration_constant
                     from _risk_affected_pipeline.diarrheal_diseases.incidence_rate
                  - AttributePipeline
                     from risk_effect.child_wasting_on_cause.diarrheal_diseases.incidence_rate
post-processors 1
                  1. rescale_post_processor

The second modifier is the effect of child wasting. The `RiskEffect` component registers
its relative risk *pipeline* as the modifier (by name, rather than as a function), which
is why it shows up as `AttributePipeline` rather than as a method name. The component it
came from, `risk_effect.child_wasting_on_cause.diarrheal_diseases.incidence_rate`, tells
you what it is.

This kind of print is a quick way to answer "what touches this value?". For instance,
here is everything that contributes to the mortality rate in our model:

In [21]:
sim_intervention.get_attribute("mortality_rate")

mortality_rate  [attribute pipeline]
registered by   mortality

source          Mortality.calculate_mortality_rate (callable)
combiner        replace_combiner
modifiers       1 (order not guaranteed)
                  - DiseaseState.adjust_mortality_rate
                     from disease_state.diarrheal_diseases
post-processors 1
                  1. rescale_post_processor

## Describing your own pipelines

Our `WastingStatus` component registered its pipeline with a `description`. It appears
near the top of the print, and the source is shown as the method that computes the
value:

In [22]:
sim_intervention.get_attribute("child_wasting.is_wasted")

child_wasting.is_wasted  [attribute pipeline]
registered by   wasting_status
description     True for simulants in the moderate or severe wasting categories (WHZ < -2)

source          WastingStatus.is_wasted (callable)
combiner        replace_combiner
modifiers       none
post-processors none

The new attribute works like any other. Comparing the two scenarios confirms that
SQ-LNS lowers the share of simulants who are wasted:

In [23]:
pd.Series(
    {
        "baseline": sim_baseline.get_population("child_wasting.is_wasted").mean(),
        "intervention": sim_intervention.get_population("child_wasting.is_wasted").mean(),
    },
    name="proportion wasted",
)

baseline        0.1138
intervention    0.1000
Name: proportion wasted, dtype: float64

## Summary

* `sim.get_attribute_names()` lists every attribute (columns and pipelines alike);
  `sim.get_population([...])` gets their values.
* `sim.get_attribute(name)` returns the pipeline object. Echo or `print()` it to see its
  source, combiner, modifiers, and post-processors; use `repr()` for the short form.
* `pipeline.source`, `pipeline.combiner`, `pipeline.mutators`, and
  `pipeline.post_processor` give you the pieces, and each prints descriptively.
* Calling `pipeline(index, mode="source")` shows the value before any modifiers, which
  is handy for checking what a modifier did.
* In your own components, pass `description=` to `register_attribute_producer`,
  `register_value_producer`, `register_attribute_modifier`, or
  `register_value_modifier` so that these prints explain themselves.